In [1]:
# WORKING DIR NEED TO BE SET BEFORE IMPORTING SETTINGS
import os

os.chdir("../..")
print("Working directory set to the root of the project")

Working directory set to the root of the project


In [2]:
import folium
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point, box

from src.settings import EPSG_WEB_MERCATOR, EPSG_WGS84

# Read data

In [3]:
access_point_df = pd.read_csv("src/data/C2C/c2corg-anonymized.2025-12-10.access_points.csv")
access_point_gdf = gpd.GeoDataFrame(
    access_point_df,
    geometry=[Point(xy) for xy in zip(access_point_df["lon"], access_point_df["lat"])],
    crs=EPSG_WGS84,
)
access_point_gdf.head()

,document_id,lon,lat,title,description,summary,geometry
0,1147027,55.364058,-21.021794,Parking des Cryptomérias,"Accès depuis Saint-Paul en 45min, route en exc...",Parking de départ pour plusieurs circuits à pi...,POINT (55.36406 -21.02179)
1,1186548,-0.358590,42.779283,Formigal - Village,NaN,Se garer à proximité de l'hôtel HG Alto Aragon...,POINT (-0.35859 42.77928)
2,1587979,170.816308,-43.858069,Fox Peak ski field,NaN,NaN,POINT (170.81631 -43.85807)
3,952546,14.510288,46.057894,Ljubljana,NaN,Ljubljana est la capitale de la Slovénie. Nomb...,POINT (14.51029 46.05789)
4,44261,7.149976,44.202556,Col de la Lombarde,[img=439892 right]col de la Lombarde[/img]\r\n,NaN,POINT (7.14998 44.20256)


In [4]:
area_gdf = gpd.read_file("src/data/transportdatagouv/contour-des-departements.geojson")
area_gdf = area_gdf.set_crs(EPSG_WGS84, allow_override=True)
area_gdf.head()

,code,nom,geometry
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.78024 46.18905, ..."
1,02,Aisne,"POLYGON ((3.17296 50.01131, 3.17382 50.01186, ..."
2,03,Allier,"POLYGON ((3.03207 46.79491, 3.03424 46.7908, 3..."
3,04,Alpes-de-Haute-Provence,"POLYGON ((5.67604 44.19143, 5.67817 44.19051, ..."
4,05,Hautes-Alpes,"POLYGON ((6.26057 45.12685, 6.26417 45.12641, ..."


In [11]:
from src.processors.c2c import C2CBusStopsProcessor

c2c_stops_gdf = C2CBusStopsProcessor.fetch(reload_pipeline=False).to_crs(EPSG_WGS84)
c2c_stops_gdf.head()

,gtfs_id,navitia_id,osm_id,name,description,line_gtfs_ids,line_osm_ids,network,network_gtfs_id,geometry,other
0,None,stop_area:OVO:S1000374,None,La Sure en Chartreuse - RD520 Village (La Sure...,None,[],[],M Réso - Pays Voironnais,None,POINT (5.65312 45.31916),"{'srid': 3857, 'stoparea_id_and_line': [{'line..."
1,None,stop_area:O38:4038505,None,RD520 Pommiers Village (La Sure en Chartreuse),None,[],[],Isère - Transisère,None,POINT (5.65302 45.31893),"{'srid': 3857, 'stoparea_id_and_line': [{'line..."
2,None,stop_area:OVO:S1002339,None,La Sure en Chartreuse - Pommiers Village (La S...,None,[],[],M Réso - Pays Voironnais,None,POINT (5.65543 45.31862),"{'srid': 3857, 'stoparea_id_and_line': [{'line..."
3,None,stop_area:OVO:S1001416,None,La Sure en Chartreuse - les Barniers (La Sure ...,None,[],[],M Réso - Pays Voironnais,None,POINT (5.65098 45.3155),"{'srid': 3857, 'stoparea_id_and_line': [{'line..."
4,None,stop_area:O38:3279412,None,Les Barniers (La Sure en Chartreuse),None,[],[],Isère - Transisère,None,POINT (5.65103 45.31545),"{'srid': 3857, 'stoparea_id_and_line': [{'line..."


In [22]:
old_c2c_stops_gdf = gpd.read_parquet("src/data/C2C/bus_stops_isere.parquet").to_crs(EPSG_WGS84)
old_c2c_stops_gdf.head()

,gtfs_id,navitia_id,osm_id,name,description,line_gtfs_ids,line_osm_ids,network,network_gtfs_id,geometry,other
0,None,stop_area:OGE:GEN15846,None,"Le Haut-Bréda, Pinsot le Village (Le Haut-Bréda)",None,[],[],Mobilités M - TouGo,None,POINT (6.09999 45.3568),"{'srid': 3857, 'stoparea_id_and_line': [{'line..."
1,None,stop_area:OGE:GEN15852,None,"Le Haut-Bréda, Hot Pic Belle Etoile (Le Haut-B...",None,[],[],Mobilités M - TouGo,None,POINT (6.09876 45.36077),"{'srid': 3857, 'stoparea_id_and_line': [{'line..."
2,None,stop_area:OGE:GEN15850,None,"Le Haut-Bréda, Chinfert (Le Haut-Bréda)",None,[],[],Mobilités M - TouGo,None,POINT (6.09766 45.36808),"{'srid': 3857, 'stoparea_id_and_line': [{'line..."
3,None,stop_area:OGE:GEN15080,None,"Le Haut-Bréda, la Piat (Le Haut-Bréda)",None,[],[],Mobilités M - TouGo,None,POINT (6.09398 45.33966),"{'srid': 3857, 'stoparea_id_and_line': [{'line..."
4,None,stop_area:OGE:GEN13054,None,"Domène, Domène Mairie (Domène)",None,[],[],Mobilités M - Tag,None,POINT (5.83816 45.20239),"{'srid': 3857, 'stoparea_id_and_line': [{'line..."


# Analysis

In [5]:
departments = ["38", "73", "74"]

In [6]:
print(f"{len(access_point_gdf)} access points in the world")

8157 access points in the world


In [7]:
france_access_point_gdf = gpd.sjoin(
    access_point_gdf,
    area_gdf.to_crs(EPSG_WGS84),
)
print(f"{len(france_access_point_gdf)} access points in France")

3707 access points in France


In [8]:
dept_access_point_gdf = gpd.sjoin(
    access_point_gdf,
    area_gdf[area_gdf["code"].isin(departments)].to_crs(EPSG_WGS84),
)
print(f"{len(dept_access_point_gdf)} access points in departments {departments}")

1366 access points in departments ['38', '73', '74']


In [13]:
print(f"{c2c_stops_gdf['navitia_id'].nunique()} C2C bus stops in the world")

1180 C2C bus stops in the world


In [14]:
france_bus_stops_gdf = gpd.sjoin(
    c2c_stops_gdf,
    area_gdf.to_crs(EPSG_WGS84),
)
print(f"{france_bus_stops_gdf['navitia_id'].nunique()} C2C bus stops in France")

1179 C2C bus stops in France


In [15]:
dept_bus_stops_gdf = gpd.sjoin(
    c2c_stops_gdf,
    area_gdf[area_gdf["code"].isin(departments)].to_crs(EPSG_WGS84),
)
print(
    f"{dept_bus_stops_gdf['navitia_id'].nunique()} C2C bus stops in departments {departments}"
)

688 C2C bus stops in departments ['38', '73', '74']


In [25]:
valid_navitia_ids = gpd.sjoin(
    c2c_stops_gdf, area_gdf[area_gdf["code"] == "38"].to_crs(EPSG_WGS84)
)["navitia_id"].drop_duplicates()
isere_bus_stops_gdf = c2c_stops_gdf[c2c_stops_gdf["navitia_id"].isin(valid_navitia_ids)]
print(f"{isere_bus_stops_gdf['navitia_id'].nunique()} C2C bus stops in Isere")

334 C2C bus stops in Isere


In [26]:
valid_navitia_ids = gpd.sjoin(
    old_c2c_stops_gdf, area_gdf[area_gdf["code"] == "38"].to_crs(EPSG_WGS84)
)["navitia_id"].drop_duplicates()
old_isere_bus_stops_gdf = old_c2c_stops_gdf[
    old_c2c_stops_gdf["navitia_id"].isin(valid_navitia_ids)
]
print(f"{old_isere_bus_stops_gdf['navitia_id'].nunique()} old C2C bus stops in Isere")

625 old C2C bus stops in Isere


# Plot data

In [39]:
# Plot activity points

# Add a base OSM map centered on Grenoble
m = folium.Map(
    location=[45.1885, 5.7245], zoom_start=9, tiles="OpenStreetMap", control_scale=True
)

# Add Waymarked Trails hiking layers
folium.TileLayer(tiles="WaymarkedTrails.hiking").add_to(m)

# Add department polygons
for dept in departments:
    dept_latlon_coords = [
        (lat, lon)
        for lon, lat in area_gdf[area_gdf["code"] == dept]["geometry"].values[0].exterior.coords
    ]
    folium.Polygon(
        locations=dept_latlon_coords,
        color="black",
        weight=3,
        fill_color="black",
        fill_opacity=0.25,
        fill=True,
    ).add_to(m)

# Add access point markers
folium.GeoJson(dept_access_point_gdf.to_json()).add_to(m)

m

In [41]:
# Plot C2C Isère bus stops

# Add a base OSM map centered on Grenoble
m = folium.Map(
    location=[45.1885, 5.7245], zoom_start=9, tiles="OpenStreetMap", control_scale=True
)

# Add Waymarked Trails hiking layers
folium.TileLayer(tiles="WaymarkedTrails.hiking", control=False).add_to(m)

# Add department polygons
for dept in ["38"]:
    dept_latlon_coords = [
        (lat, lon)
        for lon, lat in area_gdf[area_gdf["code"] == dept]["geometry"].values[0].exterior.coords
    ]
    folium.Polygon(
        locations=dept_latlon_coords,
        color="black",
        weight=3,
        fill_color="black",
        fill_opacity=0.25,
        fill=True,
    ).add_to(m)

# Add bus_stops markers
fg = folium.FeatureGroup(name=f"C2C bus stops in Isère 5th May 2025", show=True)
folium.GeoJson(
    old_isere_bus_stops_gdf[["navitia_id", "name", "network", "geometry"]].to_json(),
    marker=folium.Marker(icon=folium.Icon(icon="bus", prefix="fa", color="red")),
).add_to(fg)
fg.add_to(m)
fg = folium.FeatureGroup(name=f"C2C bus stops in Isère 10th December 2025", show=True)
folium.GeoJson(
    isere_bus_stops_gdf[["navitia_id", "name", "network", "geometry"]].to_json(),
    marker=folium.Marker(icon=folium.Icon(icon="bus", prefix="fa", color="blue")),
).add_to(fg)
fg.add_to(m)


# Add layer control to filter layers
folium.LayerControl(collapsed=False).add_to(m)

m

In [42]:
m.save(f"c2c_bus_stops_isere.html")